# 自动微分
:label:`sec_autograd`

正如 :numref:`sec_calculus`中所说，求导是几乎所有深度学习优化算法的关键步骤。
虽然求导的计算很简单，只需要一些基本的微积分。
但对于复杂的模型，手工进行更新是一件很痛苦的事情（而且经常容易出错）。

深度学习框架通过自动计算导数，即*自动微分*（automatic differentiation）来加快求导。
实际中，根据设计好的模型，系统会构建一个*计算图*（computational graph），
来跟踪计算是哪些数据通过哪些操作组合起来产生输出。
自动微分使系统能够随后反向传播梯度。
这里，*反向传播*（backpropagate）意味着跟踪整个计算图，填充关于每个参数的偏导数。

## 一个简单的例子

作为一个演示例子，(**假设我们想对函数$y=2\mathbf{x}^{\top}\mathbf{x}$关于列向量$\mathbf{x}$求导**)。
首先，我们创建变量`x`并为其分配一个初始值。


In [37]:
import torch

x = torch.arange(4.0)
x

tensor([0., 1., 2., 3.])

[**在我们计算$y$关于$\mathbf{x}$的梯度之前，需要一个地方来存储梯度。**]
重要的是，我们不会在每次对一个参数求导时都分配新的内存。
因为我们经常会成千上万次地更新相同的参数，每次都分配新的内存可能很快就会将内存耗尽。
注意，一个标量函数关于向量$\mathbf{x}$的梯度是向量，并且与$\mathbf{x}$具有相同的形状。


In [38]:
x.requires_grad_(True)  # 等价于x=torch.arange(4.0,requires_grad=True)
x.grad  # 默认值是None

(**现在计算$y$。**)


In [39]:
y = 2 * torch.dot(x, x)
y

tensor(28., grad_fn=<MulBackward0>)

`x`是一个长度为4的向量，计算`x`和`x`的点积，得到了我们赋值给`y`的标量输出。
接下来，[**通过调用反向传播函数来自动计算`y`关于`x`每个分量的梯度**]，并打印这些梯度。


In [40]:
y.backward()
x.grad

tensor([ 0.,  4.,  8., 12.])

函数$y=2\mathbf{x}^{\top}\mathbf{x}$关于$\mathbf{x}$的梯度应为$4\mathbf{x}$。
让我们快速验证这个梯度是否计算正确。


In [41]:
x.grad == 4 * x

tensor([True, True, True, True])

[**现在计算`x`的另一个函数。**]


In [42]:
# 在默认情况下，PyTorch会累积梯度，我们需要清除之前的值
x.grad.zero_()
y = x.sum() # y表示x的元素和的函数  y = x1 + x2 + ...
y.backward()
x.grad

tensor([1., 1., 1., 1.])

## 非标量变量的反向传播

当`y`不是标量时，向量`y`关于向量`x`的导数的最自然解释是一个矩阵。
对于高阶和高维的`y`和`x`，求导的结果可以是一个高阶张量。

然而，虽然这些更奇特的对象确实出现在高级机器学习中（包括[**深度学习中**]），
但当调用向量的反向计算时，我们通常会试图计算一批训练样本中每个组成部分的损失函数的导数。
这里(**，我们的目的不是计算微分矩阵，而是单独计算批量中每个样本的偏导数之和。**)


In [43]:
# 对非标量调用backward需要传入一个gradient参数，该参数指定微分函数关于self的梯度。
# 本例只想求偏导数的和，所以传递一个1的梯度是合适的
x.grad.zero_()
y = x * x
# 等价于y.backward(torch.ones(len(x)))
y.sum().backward()
x.grad

tensor([0., 2., 4., 6.])

## 分离计算

有时，我们希望[**将某些计算移动到记录的计算图之外**]。
例如，假设`y`是作为`x`的函数计算的，而`z`则是作为`y`和`x`的函数计算的。
想象一下，我们想计算`z`关于`x`的梯度，但由于某种原因，希望将`y`视为一个常数，
并且只考虑到`x`在`y`被计算后发挥的作用。

这里可以分离`y`来返回一个新变量`u`，该变量与`y`具有相同的值，
但丢弃计算图中如何计算`y`的任何信息。
换句话说，梯度不会向后流经`u`到`x`。
因此，下面的反向传播函数计算`z=u*x`关于`x`的偏导数，同时将`u`作为常数处理，
而不是`z=x*x*x`关于`x`的偏导数。


In [44]:
x.grad.zero_()
y = x * x
u = y.detach()
z = u * x

z.sum().backward()
x.grad == u

tensor([True, True, True, True])

由于记录了`y`的计算结果，我们可以随后在`y`上调用反向传播，
得到`y=x*x`关于的`x`的导数，即`2*x`。


In [45]:
x.grad.zero_()
y.sum().backward()
x.grad == 2 * x

tensor([True, True, True, True])

## Python控制流的梯度计算

使用自动微分的一个好处是：
[**即使构建函数的计算图需要通过Python控制流（例如，条件、循环或任意函数调用），我们仍然可以计算得到的变量的梯度**]。
在下面的代码中，`while`循环的迭代次数和`if`语句的结果都取决于输入`a`的值。


In [46]:
def f(a):
    b = a * 2
    while b.norm() < 1000:
        b = b * 2
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c

让我们计算梯度。


In [47]:
a = torch.randn(size=(), requires_grad=True)
d = f(a)
d.backward()

我们现在可以分析上面定义的`f`函数。
请注意，它在其输入`a`中是分段线性的。
换言之，对于任何`a`，存在某个常量标量`k`，使得`f(a)=k*a`，其中`k`的值取决于输入`a`，因此可以用`d/a`验证梯度是否正确。
* 只有纯乘法链，梯度才等于 d/a！ 因为上面的操作都是乘法，所以a.grad == d / a，就是导数/梯度/斜率

In [48]:
a.grad == d / a

tensor(True)

## 小结

* 深度学习框架可以自动计算导数：我们首先将梯度附加到想要对其计算偏导数的变量上，然后记录目标值的计算，执行它的反向传播函数，并访问得到的梯度。

## 练习

1. 为什么计算二阶导数比一阶导数的开销要更大？
1. 在运行反向传播函数之后，立即再次运行它，看看会发生什么。
1. 在控制流的例子中，我们计算`d`关于`a`的导数，如果将变量`a`更改为随机向量或矩阵，会发生什么？
1. 重新设计一个求控制流梯度的例子，运行并分析结果。
1. 使$f(x)=\sin(x)$，绘制$f(x)$和$\frac{df(x)}{dx}$的图像，其中后者不使用$f'(x)=\cos(x)$。


## 练习解答

### 1. 为什么计算二阶导数比一阶导数的开销要更大？

**原因分析：**

一阶导数只需要一次反向传播，遍历计算图一次。而二阶导数需要：

1. **计算图保留**：计算一阶导数后，需要保留计算图以计算二阶导数（通常需要 `retain_graph=True`）
2. **多次反向传播**：二阶导数需要对一阶导数再求导，相当于再遍历一次计算图
3. **内存占用更大**：需要存储一阶导数的计算图，内存开销翻倍
4. **雅可比矩阵**：对于向量输入，二阶导数是海森矩阵（Hessian），大小为 $n \times n$，远大于一阶梯度的 $n$

**具体开销对比：**

| 操作 | 计算图遍历次数 | 内存占用 | 输出大小 |
|------|---------------|---------|---------|
| 一阶导数 | 1次 | 存储原始计算图 | 向量 $n$ |
| 二阶导数 | 2次 | 存储原始+一阶计算图 | 矩阵 $n \times n$ |

---

### 2. 在运行反向传播函数之后，立即再次运行它，看看会发生什么？

**结果：报错 RuntimeError**

错误信息：`Trying to backward through the graph a second time, but the buffers have already been freed.`

**原因：** 
- 第一次 `backward()` 后，PyTorch 会释放计算图以节省内存
- 第二次调用时，计算图已经不存在，无法再次反向传播

**解决方法：**
- 方法1：`y.backward(retain_graph=True)` 保留计算图
- 方法2：重新构建计算图，重新计算 `y`

---

### 3. 将变量 `a` 更改为随机向量或矩阵，会发生什么？

**问题：** 原代码中 `a` 是标量，`d = f(a)` 也是标量，可以直接 `backward()`。如果改为向量，`d` 变成向量，无法直接 `backward()`。

**解决方法：**
- 方法1：`d.sum().backward()` 先求和变标量
- 方法2：`d.backward(torch.ones_like(d))` 传入全1权重

---

### 4. 重新设计一个求控制流梯度的例子

**例子：根据输入值选择不同的函数路径**

```python
def dynamic_function(x):
    if x < 0:
        y = x ** 2      # 路径1：梯度 = 2x
    elif x < 1:
        y = x ** 3      # 路径2：梯度 = 3x²
    else:
        y = x + 10      # 路径3：梯度 = 1
    return y
```

不同输入走不同路径，梯度自动正确计算：
- x = -2：走路径1，梯度 = -4 ✓
- x = 0.5：走路径2，梯度 = 0.75 ✓
- x = 3：走路径3，梯度 = 1 ✓

---

### 5. 绘制 $f(x) = \sin(x)$ 和其导数图像（不使用 $\cos(x)$）

**思路：** 使用自动微分逐点计算导数

```python
import torch
import matplotlib.pyplot as plt

x_vals = torch.linspace(-2*torch.pi, 2*torch.pi, 100)
y_vals = torch.sin(x_vals)

# 用自动微分计算导数
grad_vals = []
for x_val in x_vals:
    x = torch.tensor(x_val.item(), requires_grad=True)
    y = torch.sin(x)
    y.backward()
    grad_vals.append(x.grad.item())

# 绘图
plt.plot(x_vals.numpy(), y_vals.numpy(), 'b-', label='sin(x)')
plt.plot(x_vals.numpy(), grad_vals, 'r--', label='df/dx (自动微分)')
plt.legend()
plt.show()
```

结果：自动微分计算的导数与理论值 $\cos(x)$ 完全一致。

[Discussions](https://discuss.d2l.ai/t/1759)
